# AI-assisted partial verification
Executed selected existing cleaning cells on 27 October 2026. Original data only. Stops at missing v2 manifest. This is not original run evidence or a completed cleaning notebook.


In [1]:
from pathlib import Path
from collections import Counter, defaultdict
import csv, hashlib, math, random, shutil
import numpy as np
from PIL import Image, ImageDraw
import imagehash
import matplotlib.pyplot as plt

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'data/1/train/images').is_dir()), None)
assert ROOT is not None, 'Open this notebook inside the repository containing data/1.'
RAW = ROOT / 'data/1'
SPLITS = ('train', 'valid', 'test')
WRITE_COPY = False
OUTPUT = ROOT / 'data/cleaned_notebook'  # I keep this separate from data/cleaned
print('Repository:', ROOT.name)


Repository: SemProject_MLEngine


In [2]:
EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
records = []
pairing_errors = []
for split in SPLITS:
    images = sorted(p for p in (RAW / split / 'images').iterdir()
                    if p.suffix.lower() in EXTENSIONS)
    labels = {p.stem: p for p in (RAW / split / 'labels').glob('*.txt')}
    stems = Counter(p.stem for p in images)
    pairing_errors += [(split, 'ambiguous image stem', s) for s,n in stems.items() if n > 1]
    pairing_errors += [(split, 'orphan label', s) for s in labels if s not in stems]
    for p in images:
        label = labels.get(p.stem)
        if label is None:
            pairing_errors.append((split, 'missing label', p.name))
        records.append(dict(image=p.name, original_split=split,
                            source=p.stem.split('.rf.')[0], path=p, label=label))
    print(split, len(images), 'images,', len(labels), 'labels')
assert not pairing_errors, pairing_errors[:20]
assert len({r['image'] for r in records}) == len(records), 'Ambiguous manifest image names'
by_name = {r['image']: r for r in records}
print('Images:', len(records), '| source groups:', len({r['source'] for r in records}))


train 3345 images, 3345 labels
valid 397 images, 397 labels
test 198 images, 198 labels
Images: 3940 | source groups: 834


In [3]:
def read_boxes(path):
    boxes = []
    for line_number, line in enumerate(path.read_text().splitlines(), 1):
        if not line.strip():
            continue
        values = line.split()
        assert len(values) == 5, f'{path.name}:{line_number}: expected 5 fields'
        cls, x, y, w, h = map(float, values)
        assert all(math.isfinite(v) for v in (cls,x,y,w,h)), 'Nonfinite value'
        assert cls == 0, 'Expected pothole class 0'
        assert 0 <= x <= 1 and 0 <= y <= 1 and 0 < w <= 1 and 0 < h <= 1
        eps = 1e-6
        assert x-w/2 >= -eps and y-h/2 >= -eps
        assert x+w/2 <= 1+eps and y+h/2 <= 1+eps
        boxes.append((x,y,w,h))
    return boxes

errors = []
for r in records:
    try:
        with Image.open(r['path']) as im:
            im.verify()
        with Image.open(r['path']) as im:
            im.load()
            r['size'] = im.size
        r['boxes'] = read_boxes(r['label'])
    except Exception as exc:
        errors.append((r['image'], str(exc)))
print('Invalid images/labels:', len(errors))
assert not errors, errors[:20]
print('Boxes:', sum(len(r['boxes']) for r in records))
print('Empty labels:', sum(not r['boxes'] for r in records))


Invalid images/labels: 0
Boxes: 10111
Empty labels: 0


In [4]:
# We overlay a few labels to inspect the boxes against the images.
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, r in zip(axes, random.Random(42).sample(records, 3)):
    with Image.open(r['path']) as im:
        im = im.convert('RGB')
    draw = ImageDraw.Draw(im)
    W,H = im.size
    for x,y,w,h in r['boxes']:
        draw.rectangle(((x-w/2)*W, (y-h/2)*H, (x+w/2)*W, (y+h/2)*H), outline='lime', width=3)
    ax.imshow(im); ax.set_title(r['source']); ax.axis('off')
plt.show()


In [5]:
def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()
exact = defaultdict(list)
for r in records:
    r['sha256'] = sha256(r['path'])
    exact[r['sha256']].append(r)
duplicate_sets = [v for v in exact.values() if len(v) > 1]
print('Exact duplicate sets:', len(duplicate_sets))
print('Extra copies:', sum(len(v)-1 for v in duplicate_sets))
def leaking_groups(rows, key, split_key):
    groups = defaultdict(set)
    for r in rows:
        groups[r[key]].add(r[split_key])
    return {k: v for k,v in groups.items() if len(v) > 1}
print('Original source groups spanning splits:', len(leaking_groups(records, 'source', 'original_split')))
print('Original exact hashes spanning splits:', len(leaking_groups(records, 'sha256', 'original_split')))


Exact duplicate sets: 0
Extra copies: 0
Original source groups spanning splits: 385
Original exact hashes spanning splits: 0


In [6]:
groups = sorted({r['source'] for r in records})
random.Random(42).shuffle(groups)
a, b = int(.8*len(groups)), int(.9*len(groups))
proposal = {g: ('train' if i < a else 'valid' if i < b else 'test')
            for i,g in enumerate(groups)}
print('Proposal images:', Counter(proposal[r['source']] for r in records))

def load_manifest(name):
    with (ROOT / 'data' / name).open(newline='') as f:
        rows = list(csv.DictReader(f))
    assert len(rows) == len(records)
    assert len({r['image'] for r in rows}) == len(rows)
    assert {r['image'] for r in rows} == set(by_name)
    for r in rows:
        raw = by_name[r['image']]
        assert r['source'] == raw['source']
        assert r['original_split'] == raw['original_split']
        assert int(r['annotations']) == len(raw['boxes'])
        assert r['new_split'] in SPLITS
    assert not leaking_groups(rows, 'source', 'new_split')
    return rows
v2 = load_manifest('split_manifest_v2.csv')
v3 = load_manifest('split_manifest_v3.csv')
print('Recorded v2 images:', Counter(r['new_split'] for r in v2))


Proposal images: Counter({'train': 3162, 'valid': 402, 'test': 376})
Traceback (most recent call last):
  File "/home/mohamedekaden/Documents/Codex/2026-10-27/files-pasted-by-the-user-work/work/verify_existing.py", line 14, in <module>
    with contextlib.redirect_stdout(buf):exec(compile(''.join(c['source']),f'cleaning-cell-{i}','exec'),ns)
                                         ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "cleaning-cell-10", line 22, in <module>
  File "cleaning-cell-10", line 9, in load_manifest
  File "/usr/lib64/python3.14/pathlib/__init__.py", line 771, in open
    return io.open(self, mode, buffering, encoding, errors, newline)
           ~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: '/home/mohamedekaden/3.2/AI/SemProject_MLEngine/SemProject_MLEngine/data/split_manifest_v2.csv'
